# 🚲 NeuroStartup — BikeFlow: previsão de aluguel de bicicletas
**Atividade Prática 3 · Aula IAML 08 · FMU**  
Problema: prever quantas bicicletas serão alugadas **por hora** a partir de horário, calendário e condições climáticas.

**Cliente:** empresa operadora de bicicletas compartilhadas, que precisa planejar redistribuição de bicicletas e disponibilidade das estações.  
**Dados reais:** UCI Machine Learning Repository — *Bike Sharing Dataset*, registros de Washington, D.C., entre 2011 e 2012. Licença CC BY 4.0.

> O notebook foi escrito para rodar no **Google Colab** com acesso à internet. Execute **Ambiente de execução → Executar tudo** na pasta do projeto. Não inventamos métricas: os resultados são gerados ao executar o treinamento. O conjunto de teste fica guardado até a avaliação final.

## 0. Preparação
No Colab, caso as bibliotecas não estejam disponíveis, instale executando:
```python
%pip install -q keras tensorflow scikit-learn pandas numpy matplotlib joblib ipywidgets
```
Se estiver executando localmente, use `pip install -r requirements.txt`. O Colab já inclui a maioria das bibliotecas.

In [ ]:
import os, io, json, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

SEED = 42
keras.utils.set_random_seed(SEED)
PASTA = Path.cwd()
for sub in ['data', 'graficos', 'modelos', 'assets']:
    (PASTA / sub).mkdir(exist_ok=True)
print('Keras:', keras.__version__, '| Backend:', keras.backend.backend())
print('Pasta do projeto:', PASTA)

## 1. Conhecer os dados
O alvo é `cnt` (bicicletas alugadas na hora). **Não entram como entradas** `casual` e `registered`, porque somam exatamente `cnt`: usá-las seria vazamento de informação. Também descartamos `instant` (ID), `dteday` como atributo bruto e `yr` — queremos entradas que façam sentido para uma estimativa futura. A data é usada somente para manter a ordem cronológica na avaliação.

In [ ]:
# Baixa os dados originais UCI e guarda uma cópia local. Aceita ZIPs aninhados.
URL_UCI = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
URL_ESPELHO = 'https://raw.githubusercontent.com/muditp19/UCI_Bike-sharing-dataset/master/hour.csv'
ARQUIVO = PASTA / 'data' / 'hour.csv'

def extrair_hour_csv(conteudo: bytes):
    with zipfile.ZipFile(io.BytesIO(conteudo)) as zf:
        for nome in zf.namelist():
            if nome.replace('\\', '/').lower().endswith('hour.csv'):
                return zf.read(nome)
        for nome in zf.namelist():
            if nome.lower().endswith('.zip'):
                try:
                    return extrair_hour_csv(zf.read(nome))
                except (ValueError, zipfile.BadZipFile):
                    continue
    raise ValueError('hour.csv não encontrado no arquivo UCI')

def baixar_dados():
    if ARQUIVO.exists():
        return
    erros = []
    for url in (URL_UCI, URL_ESPELHO):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=50) as resposta:
                conteudo = resposta.read()
            dados_csv = extrair_hour_csv(conteudo) if url == URL_UCI else conteudo
            ARQUIVO.write_bytes(dados_csv)
            print('Base obtida de:', url)
            return
        except Exception as e:
            erros.append(str(e))
    raise RuntimeError('Não foi possível baixar o CSV. Confira sua internet no Colab ou coloque hour.csv em data/. Erros: ' + repr(erros))

baixar_dados()
df = pd.read_csv(ARQUIVO, parse_dates=['dteday']).sort_values(['dteday', 'hr']).reset_index(drop=True)
assert {'cnt','hr','casual','registered','dteday'}.issubset(df.columns)
assert len(df) > 10_000, 'A base deve conter o histórico horário completo, não uma amostra.'
assert (df['cnt'] == df['casual'] + df['registered']).all()
print(f'Linhas reais: {len(df):,} | Colunas: {df.shape[1]} | Nulos: {df.isna().sum().sum()}')
print('Período:', df['dteday'].min().date(), 'até', df['dteday'].max().date())
display(df[['dteday', 'hr','temp','hum','weathersit','cnt']].head())

In [ ]:
# Gráfico 1 — aluguel médio em cada hora
horario = df.groupby('hr')['cnt'].mean()
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(horario.index, horario.values, marker='o', linewidth=2)
ax.set(title='Bicicletas alugadas por hora (média histórica)', xlabel='Hora', ylabel='Bicicletas / hora')
ax.set_xticks(range(0,24,2)); ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(PASTA/'graficos'/'01_demanda_por_hora.png', dpi=160); plt.show()
print(f'Conclusão 1: no histórico observado, a maior média ocorreu às {int(horario.idxmax()):02d}h, com {horario.max():.1f} aluguéis por hora.')

In [ ]:
# Gráfico 2 — aluguel médio conforme situação meteorológica
clima = df.groupby('weathersit')['cnt'].mean().sort_index()
rotulos = {1:'Bom tempo', 2:'Nublado / névoa', 3:'Chuva / neve leve', 4:'Tempo severo'}
fig, ax = plt.subplots(figsize=(9,4))
ax.bar([rotulos.get(int(i), str(i)) for i in clima.index], clima.values)
ax.set(title='Média de bicicletas alugadas por situação do tempo', ylabel='Bicicletas / hora')
ax.tick_params(axis='x', labelrotation=10)
plt.tight_layout(); plt.savefig(PASTA/'graficos'/'02_demanda_por_clima.png',dpi=160); plt.show()
print(f'Conclusão 2: a maior média ocorreu na classe meteorológica {int(clima.idxmax())} ({clima.max():.1f} bicicletas/h), enquanto a menor foi na classe {int(clima.idxmin())} ({clima.min():.1f} bicicletas/h). São associações, não causalidade demonstrada.')

### Separação correta: treino, validação e teste
Dividimos em ordem temporal: **60% treino**, **20% validação**, **20% teste**, sem embaralhar dados do futuro no passado. O teste é o *cofre*; não será consultado para escolher arquitetura. Categorias recebem one-hot encoding, números contínuos recebem padronização, tudo ajustado **apenas no treino**.

In [ ]:
CAT = ['season','mnth','hr','holiday','weekday','workingday','weathersit']
NUM = ['temp','atemp','hum','windspeed']
FEATURES = CAT + NUM
ALVO = 'cnt'
X = df[FEATURES].copy()
y = df[ALVO].astype('float32').copy()
a = int(len(df)*0.60); b = int(len(df)*0.80)
X_treino, X_valid, X_teste = X.iloc[:a], X.iloc[a:b], X.iloc[b:]
y_treino, y_valid, y_teste = y.iloc[:a], y.iloc[a:b], y.iloc[b:]
print('Treino:',len(X_treino),'| Validação:',len(X_valid),'| Teste (cofre):',len(X_teste))

pre = ColumnTransformer([
    ('categorias', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT),
    ('continuas', StandardScaler(), NUM)
], remainder='drop', sparse_threshold=0)
X_treino_e = pre.fit_transform(X_treino).astype('float32')
X_valid_e  = pre.transform(X_valid).astype('float32')
y_escala = StandardScaler().fit(y_treino.to_numpy().reshape(-1,1))
y_treino_e = y_escala.transform(y_treino.to_numpy().reshape(-1,1)).astype('float32')
y_valid_e  = y_escala.transform(y_valid.to_numpy().reshape(-1,1)).astype('float32')
print('Entradas após codificação:',X_treino_e.shape[1])

## 2. Baseline × redes neurais
O baseline é a regressão Ridge, modelo linear simples. Para garantir uma comparação honesta, ele aprende apenas do treino. Em seguida são testadas três arquiteturas em Keras, com Dropout e Early Stopping. A escolha será feita **pelo MAE na validação** (média de bicicletas que a previsão erra em valor absoluto). O resultado de cada tentativa vai para `experimentos.csv`.

In [ ]:
baseline = Ridge(alpha=10.0)
baseline.fit(X_treino_e, y_treino)
mae_base_valid = mean_absolute_error(y_valid, baseline.predict(X_valid_e))
print(f'Baseline (Ridge) — MAE de validação: {mae_base_valid:.2f} bicicletas/hora')

In [ ]:
ARQUITETURAS = [(16,), (32,16), (64,32)]
redes = {}
historicos = {}
registros = []

def criar_rede(camadas):
    rede = keras.Sequential(name='bikeflow')
    rede.add(keras.Input(shape=(X_treino_e.shape[1],)))
    for unidades in camadas:
        rede.add(layers.Dense(unidades, activation='relu'))
        rede.add(layers.Dropout(0.15))
    rede.add(layers.Dense(1, activation='linear'))
    rede.compile(optimizer=keras.optimizers.Adam(learning_rate=.002), loss='mse', metrics=['mae'])
    return rede

for arquitetura in ARQUITETURAS:
    keras.utils.set_random_seed(SEED)
    rede = criar_rede(arquitetura)
    parada = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                             restore_best_weights=True, min_delta=0.0001)
    h = rede.fit(X_treino_e, y_treino_e,
                 validation_data=(X_valid_e, y_valid_e),
                 epochs=100, batch_size=256, verbose=0, callbacks=[parada])
    previsto_normalizado = rede.predict(X_valid_e, verbose=0)
    previsto = y_escala.inverse_transform(previsto_normalizado).reshape(-1)
    mae = mean_absolute_error(y_valid, previsto)
    chave = '-'.join(map(str, arquitetura))
    redes[chave] = rede
    historicos[chave] = h.history
    registros.append({'arquitetura':str(arquitetura), 'camadas':chave,
                      'epocas':len(h.history['loss']), 'mae_validacao_bikes':round(mae,2)})
    print(f'Rede {arquitetura}: {len(h.history["loss"])} épocas | MAE validação {mae:.2f} bicicletas/h')

experimentos = pd.DataFrame(registros).sort_values('mae_validacao_bikes').reset_index(drop=True)
experimentos.to_csv(PASTA/'experimentos.csv', index=False)
display(experimentos)
print('CSV criado:',PASTA/'experimentos.csv')

In [ ]:
# Curvas de aprendizado (3 redes): valores em escala padronizada do alvo
fig, axs = plt.subplots(len(historicos),1,figsize=(10,9))
for ax, (nome,h) in zip(axs, historicos.items()):
    ax.plot(h['loss'], label='Treino', linewidth=2)
    ax.plot(h['val_loss'], label='Validação', linewidth=2)
    ax.set(title='Rede '+nome, xlabel='Épocas', ylabel='Perda (MSE normalizado)')
    ax.grid(alpha=.2); ax.legend()
plt.tight_layout(); plt.savefig(PASTA/'graficos'/'03_curvas_aprendizado.png', dpi=160); plt.show()
print('Interpretação: Early Stopping interrompe o aprendizado quando a validação para de melhorar, reduzindo overfitting.')

In [ ]:
# Bônus: um segundo baseline (árvore de decisão simples), somente em validação
arvore = DecisionTreeRegressor(max_depth=8, min_samples_leaf=12, random_state=SEED)
arvore.fit(X_treino_e, y_treino)
mae_arvore_valid = mean_absolute_error(y_valid, arvore.predict(X_valid_e))
print(f'Árvore de decisão (bônus) — MAE de validação: {mae_arvore_valid:.2f} bicicletas/h')

melhor_chave = experimentos.iloc[0]['camadas']
melhor_rede = redes[melhor_chave]
print('Modelo selecionado APENAS com validação:',melhor_chave)

## 3. Abrir o cofre — teste **uma vez só**
O cofre ainda não foi usado. Agora avaliamos a rede escolhida e os dois baselines no conjunto reservado. A avaliação seguinte **não altera a escolha do modelo**.

In [ ]:
# ÚNICA célula que acessa o conjunto de teste.
X_teste_e = pre.transform(X_teste).astype('float32')
rede_teste = y_escala.inverse_transform(melhor_rede.predict(X_teste_e, verbose=0)).reshape(-1)
linear_teste = baseline.predict(X_teste_e)
arvore_teste = arvore.predict(X_teste_e)
mae_rede_teste = mean_absolute_error(y_teste, rede_teste)
mae_linear_teste = mean_absolute_error(y_teste, linear_teste)
mae_arvore_teste = mean_absolute_error(y_teste, arvore_teste)
rmse_rede_teste = np.sqrt(mean_squared_error(y_teste, rede_teste))
melhoria = 100*(mae_linear_teste-mae_rede_teste)/mae_linear_teste
print(f'TESTE FINAL | Ridge: {mae_linear_teste:.1f} bicicletas/h | Rede: {mae_rede_teste:.1f} bicicletas/h | Árvore: {mae_arvore_teste:.1f} bicicletas/h')
print(f'RMSE da rede: {rmse_rede_teste:.1f} bicicletas/h')
if melhoria >= 0:
    mensagem_negocio = (f'No período futuro reservado para teste, a BikeFlow errou em média {mae_rede_teste:.1f} bicicletas por hora, '
                        f'{melhoria:.1f}% menos que a regressão linear ({mae_linear_teste:.1f}).')
else:
    mensagem_negocio = (f'No período futuro reservado para teste, a rede errou em média {mae_rede_teste:.1f} bicicletas por hora; '
                        f'a regressão linear foi melhor ({mae_linear_teste:.1f}). A rede não está pronta para substituir o baseline.')
print('FRASE DE NEGÓCIO:', mensagem_negocio)

## 4. Produto: modelo salvo e interface de previsão
O produto precisa salvar a rede **e** o pré-processamento, pois novos pedidos precisam ser transformados da mesma forma que os dados de treinamento. A interface é um protótipo didático para previsões horárias, não uma recomendação operacional pronta para produção.

In [ ]:
melhor_rede.save(PASTA/'modelos'/'bikeflow.keras')
joblib.dump(pre, PASTA/'modelos'/'preprocessador.joblib')
joblib.dump(y_escala, PASTA/'modelos'/'escala_alvo.joblib')
metricas = {'observacoes':len(df), 'separacao':'60% treino / 20% validação / 20% teste, cronológica',
            'arquitetura':melhor_chave, 'mae_teste_rede':round(mae_rede_teste,2),
            'mae_teste_ridge':round(mae_linear_teste,2), 'mae_teste_arvore':round(mae_arvore_teste,2),
            'rmse_teste_rede':round(rmse_rede_teste,2), 'melhoria_pct_vs_ridge':round(melhoria,2),
            'mensagem_negocio':mensagem_negocio}
(PASTA/'modelos'/'metricas.json').write_text(json.dumps(metricas, ensure_ascii=False, indent=2), encoding='utf-8')
modelo_servico = keras.models.load_model(PASTA/'modelos'/'bikeflow.keras')
pre_servico = joblib.load(PASTA/'modelos'/'preprocessador.joblib')
escala_servico = joblib.load(PASTA/'modelos'/'escala_alvo.joblib')
print('Modelo, escalas e métricas salvos. Arquivos em modelos/.')

In [ ]:
# Função de previsão que recebe valores coerentes com os atributos UCI.
def estimar_bicicletas(hora=17, mes=6, dia_semana=1, dia_util=True,
                      feriado=False, clima=1, estacao=2,
                      temperatura_c=25.0, sensacao_c=26.0,
                      umidade_pct=50, vento_kmh=12):
    entrada = pd.DataFrame([{
        'season':int(estacao), 'mnth':int(mes), 'hr':int(hora),
        'holiday':int(feriado), 'weekday':int(dia_semana),
        'workingday':int(dia_util), 'weathersit':int(clima),
        'temp':(float(temperatura_c)+8)/47,
        'atemp':(float(sensacao_c)+16)/66,
        'hum':float(umidade_pct)/100,
        'windspeed':float(vento_kmh)/67
    }], columns=FEATURES)
    vetor = pre_servico.transform(entrada).astype('float32')
    saida = modelo_servico.predict(vetor, verbose=0)
    valor = float(escala_servico.inverse_transform(saida)[0,0])
    estimativa = max(0, round(valor))
    print(f'🚲 Previsão: aproximadamente {estimativa} bicicletas alugadas na hora selecionada.')
    return estimativa

# Teste rápido da função com dados de entrada plausíveis
exemplo = estimar_bicicletas()

In [ ]:
# Interface interativa no Colab/Jupyter. Execute esta célula para exibir os controles.
try:
    import ipywidgets as widgets
    from IPython.display import display
    controles = {
        'hora': widgets.IntSlider(value=17, min=0, max=23, description='Hora'),
        'mes': widgets.IntSlider(value=6, min=1, max=12, description='Mês'),
        'dia_semana': widgets.Dropdown(options=[('Domingo',0),('Segunda',1),('Terça',2),('Quarta',3),('Quinta',4),('Sexta',5),('Sábado',6)], value=1, description='Dia'),
        'dia_util': widgets.Checkbox(value=True, description='Dia útil'),
        'feriado': widgets.Checkbox(value=False, description='Feriado'),
        'clima': widgets.Dropdown(options=[('Bom tempo',1),('Nublado/névoa',2),('Chuva/neve leve',3),('Tempo severo',4)], description='Clima'),
        'estacao': widgets.Dropdown(options=[('1',1),('2',2),('3',3),('4',4)],value=2,description='Estação UCI'),
        'temperatura_c': widgets.FloatSlider(value=25,min=-8,max=39,step=1,description='Temp. °C'),
        'sensacao_c': widgets.FloatSlider(value=26,min=-16,max=50,step=1,description='Sens. °C'),
        'umidade_pct': widgets.IntSlider(value=50,min=0,max=100,description='Umidade %'),
        'vento_kmh': widgets.IntSlider(value=12,min=0,max=67,description='Vento km/h')
    }
    display(widgets.VBox([widgets.HTML('<h3>🚲 BikeFlow — simule uma hora de demanda</h3>'),
                           widgets.interactive(estimar_bicicletas, **controles)]))
except ImportError:
    print('ipywidgets não instalado. Instale com %pip install ipywidgets e rode esta célula novamente.')

## 5. Documentação e ética
Os registros são históricos (2011–2012) e de uma cidade específica. Mesmo que a rede tenha bom desempenho nesse período, **não deve ser aplicada automaticamente em São Paulo ou em 2026**. Precisamos atualizar o treinamento, monitorar erro, avaliar mudanças de mobilidade e considerar condições locais.

In [ ]:
card = f"""# Model Card — BikeFlow

## Uso pretendido
Prever o número aproximado de bicicletas compartilhadas alugadas por hora, para planejamento de disponibilidade e redistribuição em cenários históricos comparáveis.

## Não usar para
Não usar para punir trabalhadores, definir preços individuais discriminatórios, negar atendimento, tomar decisões de segurança ou justificar investimentos sem revisão humana.

## Dados
UCI Bike Sharing (Capital Bikeshare, Washington, D.C., 2011–2012): **{len(df)}** registros horários reais. Atributos: {', '.join(FEATURES)}. Excluídas as colunas `casual` e `registered` para evitar vazamento de resposta.

## Modelo e avaliação
Rede neural Keras com camadas **{melhor_chave}**, Dropout (15%) e Early Stopping, comparada a regressão Ridge e árvore de decisão.
Divisão cronológica em 60% treino / 20% validação / 20% teste. Seleção feita apenas na validação.
**MAE no teste:** rede **{mae_rede_teste:.2f}** bicicletas/h; Ridge **{mae_linear_teste:.2f}**; árvore **{mae_arvore_teste:.2f}**. RMSE da rede: **{rmse_rede_teste:.2f}**.

## Limitações
Dados de 2011–2012 de Washington; não generalizar para 2026, outras cidades, alterações de hábitos ou novas estações. O dataset é histórico e observacional; a sazonalidade, a coleta e a mudança de demanda causam desvios. Avaliar justiça e impactos operacionais, validar dados de entrada, acompanhar o erro e realizar re-treinamento com dados atuais. Esta demonstração não utiliza disponibilidade de bicicletas por estação, trânsito, eventos nem previsão meteorológica.

## Fonte e licença
Fanaee-T, H. (2013). *Bike Sharing*. UCI Machine Learning Repository. https://doi.org/10.24432/C5W894 — CC BY 4.0.
"""
(PASTA/'MODEL_CARD.md').write_text(card,encoding='utf-8')
(PASTA/'RESULTADOS.md').write_text('# Resultados executados — BikeFlow\n\n'+mensagem_negocio +
    f'\n\n| Modelo | MAE teste (bicicletas/h) |\n|---|---:|\n| Ridge | {mae_linear_teste:.2f} |\n| Rede Keras | {mae_rede_teste:.2f} |\n| Árvore | {mae_arvore_teste:.2f} |\n',encoding='utf-8')
print('MODEL_CARD.md e RESULTADOS.md gerados com números REAIS do teste executado.')
print('Para cumprir o item “print da tela” da rubrica, tire uma captura da interface acima e salve em assets/tela_funcionando.png. Atualize a imagem correspondente no README.')

### ✅ Checklist de entrega
- [ ] Células todas executadas, com os gráficos e resultados aparecendo no `.ipynb` salvo.
- [ ] `experimentos.csv` gerado (três redes).
- [ ] `modelos/bikeflow.keras`, `modelos/preprocessador.joblib` e `modelos/escala_alvo.joblib` salvos.
- [ ] Gráficos `01`, `02` e `03` gerados.
- [ ] `MODEL_CARD.md` e `RESULTADOS.md` atualizados.
- [ ] Captura **real** da interface funcionando salva em `assets/tela_funcionando.png`.
- [ ] Nome e RA de todos os integrantes no README, declaração de uso da IA e repositório público no GitHub.

**Como capturar o estado executado no Colab:** Arquivo → Fazer download → Fazer download de .ipynb. Para salvar arquivos gerados, compacte a pasta do projeto ou baixe os arquivos pelo painel lateral.